# Práctica guiada — De formulario a experiencia de inferencia

En S5 ya construisteis un formulario Streamlit que recoge los 11 atributos y llama a un gateway basado en el contrato de S4. En S6 **no se rediseña el contrato ni se vuelve a implementar el formulario**. El objetivo es convertir esa pantalla funcional en una interfaz operable: mantiene el resultado al haber un rerun, hace visible lo que está ocurriendo y enseña la información justa para interpretar e investigar una predicción.

Trabajad en parejas. Completad los `TODO`, ejecutad las comprobaciones y conservad el notebook como evidencia para el taller. La implementación queda para la Clase 2.

## Entrega de la práctica

Al terminar debéis poder enseñar cuatro decisiones concretas:

1. qué permanece en la sesión, qué se cachea y qué no se retiene;
2. la máquina `idle → loading → success/error`;
3. un plano de pantalla con componentes Streamlit concretos;
4. mensajes seguros para confianza baja, lentitud y error.

No se entrega código de inferencia nuevo. S4 sigue siendo la única frontera de validación y predicción.

## 1. Comprobar el punto de partida de S5

Ejecutad la celda. Debe demostrar que partís del formulario y gateway de S5, no de una segunda implementación. Después anotad una limitación que observaríais al cambiar un widget o al enviar una segunda predicción.

In [1]:
from pathlib import Path

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'semana5/modules/05-streamlit-basic-model-ui').is_dir()
)
s5_app = repo_root / 'semana5/modules/05-streamlit-basic-model-ui/solutions/02-first-streamlit/app.py'
source = s5_app.read_text(encoding='utf-8')

checks = {
    'reutiliza st.form': 'st.form(' in source,
    'recoge valores antes de inferir': 'collect_values' in source,
    'aún no usa session_state': 'st.session_state' not in source,
}
print('app de S5:', s5_app)
for label, passed in checks.items():
    print(f"{'✓' if passed else '✗'} {label}")
assert all(checks.values())

limitacion_s5 = 'El resultado solo está en una variable local: en el siguiente rerun desaparece y no se puede reintentar ni limpiar.'
print('Limitación observada:', limitacion_s5)

app de S5: C:\Users\Byadr\Documents\Master\operacion_modelos\icai-muiaap-operacion-de-modelos\semana5\modules\05-streamlit-basic-model-ui\solutions\02-first-streamlit\app.py
✓ reutiliza st.form
✓ recoge valores antes de inferir
✓ aún no usa session_state
Limitación observada: El resultado solo está en una variable local: en el siguiente rerun desaparece y no se puede reintentar ni limpiar.


## 2. Decidir dónde vive cada cosa

Un rerun vuelve a ejecutar el script. Una variable local desaparece; `st.session_state` conserva información de **esa sesión**; `st.cache_resource` reutiliza recursos como un gateway o modelo. Un recurso cacheado global puede compartirse entre usuarios, así que debe ser seguro para concurrencia.

Completad la tabla y justificad cada decisión. No guardéis secretos ni las 11 features en la telemetría. Los valores del último envío solo se conservan para permitir un reintento y no se muestran ni se registran.

| Clave o recurso | Dónde vive | ¿Sobrevive a un rerun? | ¿Se limpia al pulsar «Limpiar»? | Motivo |
| --- | --- | --- | --- | --- |
| `last_state` | st.session_state | Sí | Sí, vuelve a idle | Es lo último que ve la persona y tiene que seguir visible tras un rerun. |
| `last_values` | st.session_state | Sí | Sí, se vacía | Solo sirve para reintentar el último envío. No se muestra ni se registra. |
| `telemetry` | st.session_state | Sí | No | Son contadores de la sesión y no se deben perder al limpiar. |
| gateway / bundle | st.cache_resource | Sí, se carga una vez | No | Cargar el bundle es caro y es igual para todos los usuarios. |
| respuesta de una petición | Variable local, se guarda dentro de last_state | No como variable suelta | Sí | Es de una sola persona, no se puede cachear como recurso compartido. |

Referencias: [`st.session_state`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state) y [`st.cache_resource`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.cache_resource).

## 3. Diseñar la pantalla, no solo el estado

La interfaz debe tener cuatro zonas, en este orden:

1. **acción:** el formulario heredado de S5;
2. **operación:** qué está ocurriendo mientras se ejecuta;
3. **resultado:** categoría, confianza y latencia;
4. **trazabilidad:** versiones y `request_id`, visibles solo si se necesitan.

Elegid componentes que tengan una función. `st.columns` organiza, `st.status` comunica una operación, `st.metric` destaca un dato y `st.expander` aplaza el detalle técnico. No uséis una barra de progreso para la confianza: representa avance de una tarea, no certeza del modelo.

In [2]:
# Sustituid cada TODO por el componente y la justificación que usaríais.
# Las claves describen zonas de la interfaz; no son nuevos contratos de ML.

ui_plan = {
    'formulario': {'component': 'st.form', 'reason': 'Es el formulario de S5 y solo infiere al pulsar el botón.'},
    'estado_operacion': {'component': 'st.status', 'reason': 'Enseña que la inferencia se está ejecutando y cómo terminó.'},
    'resumen_resultado': {'component': 'st.metric', 'reason': 'Categoría, confianza y latencia se leen de un vistazo.'},
    'mensaje_interpretacion': {'component': 'st.warning', 'reason': 'Avisa de confianza baja o respuesta lenta sin decir que el resultado es seguro.'},
    'trazabilidad': {'component': 'st.expander', 'reason': 'Versiones y request_id quedan ocultos hasta que se necesitan.'},
    'separacion_telemetria': {'component': 'st.divider', 'reason': 'Separa la telemetría de la sesión del resultado de la petición.'},
}

allowed_components = {
    'st.form', 'st.status', 'st.empty', 'st.columns', 'st.container',
    'st.metric', 'st.success', 'st.info', 'st.warning', 'st.error',
    'st.expander', 'st.divider', 'st.caption',
}
required_zones = {
    'formulario', 'estado_operacion', 'resumen_resultado',
    'mensaje_interpretacion', 'trazabilidad', 'separacion_telemetria',
}

assert set(ui_plan) == required_zones
for zone, decision in ui_plan.items():
    assert set(decision) == {'component', 'reason'}, zone

pending = [zone for zone, decision in ui_plan.items() if 'TODO' in decision.values()]
invalid = [
    f"{zone}: {decision['component']}"
    for zone, decision in ui_plan.items()
    if 'TODO' not in decision['component'] and decision['component'] not in allowed_components
]
print('Zonas pendientes:', pending or 'ninguna')
print('Componentes no permitidos:', invalid or 'ninguno')
print('Comprobación final: pending == [] and invalid == []')

Zonas pendientes: ninguna
Componentes no permitidos: ninguno
Comprobación final: pending == [] and invalid == []


### Plano mínimo esperado

```text
Título y contexto breve

[ formulario de S5 en dos columnas ]

[ estado de la operación ]

[ Categoría ]  [ Confianza ]  [ Latencia ]
mensaje interpretativo de confianza y latencia
▶ Trazabilidad: versiones y request_id

────────────────────────────────────
Telemetría agregada de la sesión
```

Podéis entregar este plano en el notebook o redibujarlo en draw.io. Justificad cualquier cambio respecto a esta estructura.

## 4. Máquina de estados y recuperación

Completad la tabla. Una respuesta válida con latencia superior a 300 ms sigue en `success`: añade un aviso técnico, no se convierte en error. La confianza es una señal para revisar, no un diagnóstico ni una garantía.

| Estado | Evento de entrada | Zona de pantalla que cambia | Qué se muestra | Acción permitida | Evidencia |
| --- | --- | --- | --- | --- | --- |
| `idle` | Inicio / limpiar | Zona de estado | Mensaje para completar el formulario y pulsar Ejecutar inferencia | Rellenar y enviar el formulario | No hay resultado ni error en pantalla |
| `loading` | Enviar formulario | Zona de estado | st.status con Ejecutando inferencia | Esperar | Se ve el status mientras responde el gateway |
| `success` | Payload válido | Zona de resultado | Tres métricas, mensaje de confianza y latencia y expander con versiones y request_id | Enviar otra vez o limpiar | El resultado sigue tras un rerun y la telemetría suma un éxito |
| `error` | Excepción controlada | Zona de resultado | st.error con título, mensaje, recuperación y request_id | Reintentar o limpiar | No hay traceback y la telemetría suma un error con su código |

Incluid una recuperación para entrada inválida, bundle no disponible y timeout. Indicad también qué no debe ver la persona: traceback, rutas internas, secretos o el payload completo.

Recuperación:

- Entrada inválida: se pide revisar los valores. La persona corrige y vuelve a enviar.
- Bundle no disponible: se dice que el modelo no está disponible. Se revisa MODEL_UI_BUNDLE y se reintenta.
- Timeout: se dice que la respuesta tardó demasiado. Se pulsa Reintentar con los últimos valores.

La persona no ve el traceback, rutas internas, secretos ni el payload completo.

## 5. Escribir el copy de una interfaz responsable

Cada mensaje debe decir qué ha ocurrido, qué significa y qué puede hacer la persona. No debéis prometer que una predicción es correcta. Completad textos de una frase, sin terminología interna.

In [3]:
ux_copy = {
    'low_confidence': 'La confianza es baja: toma el resultado como orientativo y revísalo antes de decidir.',
    'slow_success': 'El resultado es válido, pero la respuesta tardó más de lo esperado.',
    'invalid_input': 'Algunos valores no son válidos: revisa los campos y vuelve a enviar.',
    'artifact_unavailable': 'El modelo no está disponible ahora mismo: inténtalo de nuevo en unos minutos.',
}

assert set(ux_copy) == {
    'low_confidence', 'slow_success', 'invalid_input', 'artifact_unavailable'
}
for scenario, message in ux_copy.items():
    assert isinstance(message, str) and message.strip(), scenario

pending_copy = [scenario for scenario, message in ux_copy.items() if 'TODO' in message]
print('Mensajes pendientes:', pending_copy or 'ninguno')
print('Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?')

Mensajes pendientes: ninguno
Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?


## 6. Preparar el taller

Entregad el notebook con los TODO completados. Antes de cerrar, otra pareja debe poder responder estas preguntas leyendo vuestro trabajo:

- ¿Qué se mantiene al pulsar otro widget y qué se elimina con «Limpiar último resultado»?
- ¿Qué función cargará el gateway una sola vez y cuál será su límite?
- ¿Qué componentes dibujará `render_state()` para `loading`, `success` y `error`?
- ¿Qué ve una persona con confianza baja o una respuesta lenta?
- ¿Dónde aparecen el `request_id` y las versiones sin invadir el resultado?

En el taller se implementarán estas decisiones en `session.py`, `policies.py`, `presentation.py`, `controller.py` y `app.py`.

## Respuestas para el taller

### Contrato de sesión

| Clave | Tipo | Se conserva entre reruns | Se puede limpiar | Por qué |
| --- | --- | --- | --- | --- |
| last_state | UiState | Sí | Sí, vuelve a idle | Permite volver a pintar el último resultado tras un rerun. |
| telemetry | Telemetry | Sí | No | Acumula contadores, errores, latencias y versiones de la sesión. |
| valores completos del formulario | dict de 11 floats en last_values | Sí, solo para reintentar | Sí | No se muestran ni se guardan en la telemetría. |

En la telemetría no guardamos los valores del formulario ni secretos, porque pueden ser datos de la persona y no hacen falta para medir el servicio. Solo guardamos contadores, códigos de error, latencias y versiones.

### Caché y UX

- Con st.cache_resource se protege get_cached_gateway, que crea el gateway del bundle una sola vez. Su límite es que se comparte entre usuarios, así que no guarda nada de una petición.
- No se cachea como recurso la respuesta de una petición ni los valores del formulario.
- loading se muestra con st.status, el resultado con st.metric, las advertencias con st.warning y la trazabilidad con st.expander.
- La confianza va en una métrica con un mensaje porque una barra de progreso parece el avance de una tarea y no la seguridad del modelo.
- Con confianza baja se muestra el mensaje de low_confidence.
- Bundle ausente: revisar MODEL_UI_BUNDLE y reintentar. Timeout: reintentar. Error de entrada: corregir y volver a enviar.

### Cambio en el laboratorio visual

En el escenario de éxito el aviso verde repetía la categoría, que ya sale en la métrica Resultado. Lo cambiamos por "Inferencia completada". Así la categoría se lee una sola vez y el aviso solo dice que la operación terminó.

### Plan de implementación

| Requisito | Función | Test |
| --- | --- | --- |
| Iniciar la sesión y limpiar sin perder la telemetría | initialize_session_state y clear_last_state en session.py | test_session.py |
| Umbrales de confianza y latencia | classify_confidence y classify_latency en policies.py | test_presentation.py |
| Vista del resultado y errores seguros | build_prediction_view y to_user_facing_error en presentation.py | test_presentation.py |
| loading, success o error y telemetría | PredictionController.submit en controller.py | test_controller.py |
| Gateway cacheado y dibujo de los estados | get_cached_gateway y render_state en app.py | Revisión visual |


## Debrief

- ¿Qué limita a S5 aunque el formulario ya funcione?
- ¿Por qué el gateway es un recurso cacheable y el resultado de una petición pertenece a una sesión?
- ¿Qué componente hace visible cada transición sin añadir una segunda lógica de inferencia?
- ¿Por qué categoría, confianza y latencia no tienen el mismo significado?
- ¿Qué parte de este diseño se conserva al sustituir el gateway local por HTTP en S7?